In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('/content/train.csv')
center = pd.read_csv('/content/fulfilment_center_info.csv')
meal = pd.read_csv('/content/meal_info.csv')

In [5]:
df.head(1)

,id,week,center_id,meal_id,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders
0,1379560,1,55,1885,136.83,152.29,0,0,177


In [4]:
center.head(1)

,center_id,city_code,region_code,center_type,op_area
0,11,679,56,TYPE_A,3.7


In [6]:
meal.head(1)

,meal_id,category,cuisine
0,1885,Beverages,Thai


In [9]:
df = df.join(
    meal.set_index('meal_id'), on='meal_id').join(center.set_index('center_id'), on='center_id').copy()

In [10]:
df.head()

,id,week,center_id,meal_id,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders,category,cuisine,city_code,region_code,center_type,op_area
0,1379560,1,55,1885,136.83,152.29,0,0,177,Beverages,Thai,647,56,TYPE_C,2.0
1,1466964,1,55,1993,136.83,135.83,0,0,270,Beverages,Thai,647,56,TYPE_C,2.0
2,1346989,1,55,2539,134.86,135.86,0,0,189,Beverages,Thai,647,56,TYPE_C,2.0
3,1338232,1,55,2139,339.50,437.53,0,0,54,Beverages,Indian,647,56,TYPE_C,2.0
4,1448490,1,55,2631,243.50,242.50,0,0,40,Beverages,Indian,647,56,TYPE_C,2.0


This is used for forecasting demand.

Our FinishedGood(Semiconductor) Table is modeled after the **Kaggle Food Demand Forecasting Dataset**: https://www.kaggle.com/datasets/kannanaikkal/food-demand-forecasting

In [11]:
df = df.rename(
    columns={
        'center_id': 'facility_id',
        'meal_id': 'semiconductor_id',
        'num_orders': 'customer_orders',
        'checkout_price': 'realized_selling_price',
        'base_price': 'list_price'})

In [13]:
df

,id,week,facility_id,semiconductor_id,realized_selling_price,list_price,emailer_for_promotion,homepage_featured,customer_orders,category,cuisine,city_code,region_code,center_type,op_area
0,1379560,1,55,1885,136.83,152.29,0,0,177,Beverages,Thai,647,56,TYPE_C,2.0
1,1466964,1,55,1993,136.83,135.83,0,0,270,Beverages,Thai,647,56,TYPE_C,2.0
2,1346989,1,55,2539,134.86,135.86,0,0,189,Beverages,Thai,647,56,TYPE_C,2.0
3,1338232,1,55,2139,339.50,437.53,0,0,54,Beverages,Indian,647,56,TYPE_C,2.0
4,1448490,1,55,2631,243.50,242.50,0,0,40,Beverages,Indian,647,56,TYPE_C,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
456543,1271326,145,61,1543,484.09,484.09,0,0,68,Desert,Indian,473,77,TYPE_A,4.5
456544,1062036,145,61,2304,482.09,482.09,0,0,42,Desert,Indian,473,77,TYPE_A,4.5
456545,1110849,145,61,2664,237.68,321.07,0,0,501,Salad,Italian,473,77,TYPE_A,4.5
456546,1147725,145,61,2569,243.50,313.34,0,0,729,Salad,Italian,473,77,TYPE_A,4.5


# Date Dimension Bridge

week -> approximate date -> month/year

In [24]:
# assume week 1 = Jan 2018 (arbitrary but consistent anchor)
start_date = pd.to_datetime('2023-01-01')

df['date'] = start_date + pd.to_timedelta((df['week'] - 1) * 7, unit='D')

df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['year_month'] = df['date'].dt.to_period('M').astype(str)

# Dropping Unnecessary Columns
- Don't map cleanly to semiconductors
- Only add noise, not signal

In [22]:
df = df.drop(
    columns=[
        'category',
        'cuisine',
        'city_code',
        'region_code',
        'center_type',
        'op_area',
        'id'])

In [25]:
df

,week,facility_id,semiconductor_id,realized_selling_price,list_price,emailer_for_promotion,homepage_featured,customer_orders,date,year,month,year_month
0,1,55,1885,136.83,152.29,0,0,177,2023-01-01,2023,1,2023-01
1,1,55,1993,136.83,135.83,0,0,270,2023-01-01,2023,1,2023-01
2,1,55,2539,134.86,135.86,0,0,189,2023-01-01,2023,1,2023-01
3,1,55,2139,339.50,437.53,0,0,54,2023-01-01,2023,1,2023-01
4,1,55,2631,243.50,242.50,0,0,40,2023-01-01,2023,1,2023-01
...,...,...,...,...,...,...,...,...,...,...,...,...
456543,145,61,1543,484.09,484.09,0,0,68,2025-10-05,2025,10,2025-10
456544,145,61,2304,482.09,482.09,0,0,42,2025-10-05,2025,10,2025-10
456545,145,61,2664,237.68,321.07,0,0,501,2025-10-05,2025,10,2025-10
456546,145,61,2569,243.50,313.34,0,0,729,2025-10-05,2025,10,2025-10


# Forecasting
We need **Lead-time aware demand coverage**

We are forecasting **demand over supplier lead-time horizon**

For our model: *We forecast demand over the next 16-20 weeks to ensure coverage across all suppliers*

**We train on 135 weeks, validate on last 10 weeks of "training" data**
This proves the model **generalizes to unseen future demand**
 - we validate on holdout future periods to ensure forecasting reliability

 **Procureent decisions are lead-time forward-looking, not short-term reactive**

# Step 1 - Model Validation (for Presentation)
Split:
 - Train: weeks 1-135
 - Validate: weeks 136-145

 Show:
 - error metrics
 - plots
 - model able to generalize to unseen future

 **Credibility in forecasting demand aspect of agent**

In [37]:
df

,week,facility_id,semiconductor_id,realized_selling_price,list_price,emailer_for_promotion,homepage_featured,customer_orders,date,year,month,year_month
0,1,55,1885,136.83,152.29,0,0,177,2023-01-01,2023,1,2023-01
1,1,55,1993,136.83,135.83,0,0,270,2023-01-01,2023,1,2023-01
2,1,55,2539,134.86,135.86,0,0,189,2023-01-01,2023,1,2023-01
3,1,55,2139,339.50,437.53,0,0,54,2023-01-01,2023,1,2023-01
4,1,55,2631,243.50,242.50,0,0,40,2023-01-01,2023,1,2023-01
...,...,...,...,...,...,...,...,...,...,...,...,...
456543,145,61,1543,484.09,484.09,0,0,68,2025-10-05,2025,10,2025-10
456544,145,61,2304,482.09,482.09,0,0,42,2025-10-05,2025,10,2025-10
456545,145,61,2664,237.68,321.07,0,0,501,2025-10-05,2025,10,2025-10
456546,145,61,2569,243.50,313.34,0,0,729,2025-10-05,2025,10,2025-10


In [41]:
df.shape

(456548, 12)

**We have demand history up to early October 2025, and we are now making a procurement decision in December 2025 using the most recent available data**

In real companies:
 - Demand data is lagged
 - Reporting pipelines are not real-time
 - Decisions are made using latest available snapshot

**OUR TIMELINE:**
- Historical demand data:
  - Jan 2023 -> Oct 5 2025 (observed)
- Decision point:
  - Dec 1, 2025 ("today")
- Forecast horizon:
  - Dec 2025 -> Apr 2026 (next 16-20 weeks)

# Fixing Structure of Demand Inventory Data

**Goal we want:**
- Facilities: 2-4
- Semiconductor SKUs: 8-12

In [35]:
df.facility_id.nunique()

77

In [36]:
df.semiconductor_id.nunique()

51

## Selecting/Filtering Top Facilities by volume

In [38]:
facility_volume = df.groupby('facility_id')['customer_orders'].sum()

In [40]:
top_facilities = (facility_volume.sort_values(ascending=False).head(4).index)

In [42]:
df = df[df['facility_id'].isin(top_facilities)]

In [43]:
df.shape

(28024, 12)

## Selecting/Filtering Top Semiconductors by Volume

In [44]:
sku_volume = df.groupby("semiconductor_id").customer_orders.sum()

top_skus = (
    sku_volume.sort_values(ascending=False).head(12).index)

In [46]:
df = df[df['semiconductor_id'].isin(top_skus)]

In [47]:
# reset index
df = df.reset_index(drop=True)

In [48]:
df.shape

(6959, 12)

We've selected PRODUCTS WITH REAL DEMAND and FACILITIES WITH REAL ACTIVITY

No dead SKUs, sparse demand, or noise issues.

**This data will eventually become the fact_semiconductor_demand table**

In [49]:
df.facility_id.nunique()

4

In [50]:
df.semiconductor_id.nunique()

12

Why matters downstream:
1. Forecasting
 - cleaner patterns
 - less noise
 - faster training

2. BOM (Bill of Materials)
 - Manageable mapping
 - interpretable component demand

3. LP Optimization
 - realistic supplier allocation
 - explainable results

# Relabeling SKUs and Facility_IDs

In [51]:
df.semiconductor_id.unique()

array([1885, 1993, 2539, 1778, 1062, 2707, 2290, 1727, 1109, 2826, 1754,
       1971])

In [52]:
df.facility_id.unique()

array([13, 52, 10, 43])

In [54]:
sku_map = {
    old_id: f"SEMICONDUCTOR_{i+1}"
    for i, old_id in enumerate(top_skus)}

facility_map = {
    old_id: f"FACILITY_{i+1}"
    for i, old_id in enumerate(top_facilities)}

df = df.assign(
    semiconductor_id=lambda x: x.semiconductor_id.map(sku_map),
    facility_id=lambda x: x.facility_id.map(facility_map))

In [56]:
df

,week,facility_id,semiconductor_id,realized_selling_price,list_price,emailer_for_promotion,homepage_featured,customer_orders,date,year,month,year_month
0,1,FACILITY_1,SEMICONDUCTOR_2,135.86,122.28,0,1,2132,2023-01-01,2023,1,2023-01
1,1,FACILITY_1,SEMICONDUCTOR_5,134.86,122.28,0,1,2418,2023-01-01,2023,1,2023-01
2,1,FACILITY_1,SEMICONDUCTOR_11,133.86,133.86,0,0,474,2023-01-01,2023,1,2023-01
3,1,FACILITY_1,SEMICONDUCTOR_12,184.36,182.36,0,0,513,2023-01-01,2023,1,2023-01
4,1,FACILITY_1,SEMICONDUCTOR_9,185.33,185.33,0,0,998,2023-01-01,2023,1,2023-01
...,...,...,...,...,...,...,...,...,...,...,...,...
6954,145,FACILITY_2,SEMICONDUCTOR_7,464.63,464.63,0,0,555,2025-10-05,2025,10,2025-10
6955,145,FACILITY_2,SEMICONDUCTOR_3,283.30,283.30,0,0,1796,2025-10-05,2025,10,2025-10
6956,145,FACILITY_2,SEMICONDUCTOR_8,377.33,379.33,0,0,661,2025-10-05,2025,10,2025-10
6957,145,FACILITY_2,SEMICONDUCTOR_6,310.40,312.40,0,0,1147,2025-10-05,2025,10,2025-10


In [57]:
df.groupby(
    ['facility_id', 'semiconductor_id']
).size().describe()

,0
count,48.000000
mean,144.979167
std,0.144338
min,144.000000
25%,145.000000
50%,145.000000
75%,145.000000
max,145.000000


In [59]:
(df.realized_selling_price <= df.list_price).mean()

np.float64(0.7311395315418882)

# Attemtping to Scale Price Attributes to bring them up to the range of actual wholesale selling prices for semiconductors

In [60]:
df.realized_selling_price.describe()

,realized_selling_price
count,6959.000000
mean,237.314916
std,89.198848
min,98.970000
25%,158.110000
50%,224.100000
75%,301.685000
max,466.630000


Something like 2K - 8K (USD) is much more believable for semiconducot components.

In [61]:
# computing scaling factor
current_mean = df.realized_selling_price.mean()
target_mean = 4000

scale_factor = target_mean / current_mean

In [62]:
# applying scaling
df['realized_selling_price'] *= scale_factor
df['list_price'] *= scale_factor

In [63]:
# still holds. CHECK
(df.realized_selling_price <= df.list_price).mean()

np.float64(0.7311395315418882)

Price

In [65]:
df

,week,facility_id,semiconductor_id,realized_selling_price,list_price,emailer_for_promotion,homepage_featured,customer_orders,date,year,month,year_month
0,1,FACILITY_1,SEMICONDUCTOR_2,2289.952984,2061.058817,0,1,2132,2023-01-01,2023,1,2023-01
1,1,FACILITY_1,SEMICONDUCTOR_5,2273.097744,2061.058817,0,1,2418,2023-01-01,2023,1,2023-01
2,1,FACILITY_1,SEMICONDUCTOR_11,2256.242503,2256.242503,0,0,474,2023-01-01,2023,1,2023-01
3,1,FACILITY_1,SEMICONDUCTOR_12,3107.432152,3073.721671,0,0,513,2023-01-01,2023,1,2023-01
4,1,FACILITY_1,SEMICONDUCTOR_9,3123.781736,3123.781736,0,0,998,2023-01-01,2023,1,2023-01
...,...,...,...,...,...,...,...,...,...,...,...,...
6954,145,FACILITY_2,SEMICONDUCTOR_7,7831.450428,7831.450428,0,0,555,2025-10-05,2025,10,2025-10
6955,145,FACILITY_2,SEMICONDUCTOR_3,4775.089655,4775.089655,0,0,1796,2025-10-05,2025,10,2025-10
6956,145,FACILITY_2,SEMICONDUCTOR_8,6359.987926,6393.698407,0,0,661,2025-10-05,2025,10,2025-10
6957,145,FACILITY_2,SEMICONDUCTOR_6,5231.866674,5265.577155,0,0,1147,2025-10-05,2025,10,2025-10


Different finished semiconductor SKUs have different price tiers due to differences in complexity, performance, and quality level. We preserved those relative differences when adapting the demand data.
 - some SKUs are premium / higher-performance chips
 - some are lower-cost / standard chips

 We preserved underlying relative pricing relationships while scaling values into a more realistic semiconductor range.

**WHY ACCEPTABLE**
- We repurposed this demand dataset
- Preserved relative structure instead of inventing random prices
- Project's core value is procurement decision logic, not synthetically deriving down to the dollar exact finished-chip pricing

**DO NOT FORGET**
- User parameters going into scoring (factoring in/how it does):
  - lambda
  - quantity
  - compliance threshold

In [66]:
df.to_csv('finished_goods_demand_table.csv', index=False)